In [ ]:
#RFM Creation
orders['order_date'] = pd.to_datetime(orders['order_date'])
snapshot_date = pd.to_datetime("2025-09-30")

rfm = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'order_value': 'sum',
    'order_date': 'max'
}).reset_index()

rfm.columns = ['customer_id', 'frequency', 'monetary', 'last_order_date']
rfm['recency'] = (snapshot_date - rfm['last_order_date']).dt.days


In [ ]:
#Add Extra Signals
# Support tickets
support_count = support.groupby('customer_id').size().reset_index(name='tickets')



In [ ]:
# Web activity
web_activity = web[['customer_id','sessions_30d']]

rfm = rfm.merge(support_count, on='customer_id', how='left')
rfm = rfm.merge(web_activity, on='customer_id', how='left')




In [ ]:
#Segmentation Logic - example
def segment(row):
    if row['recency'] < 30 and row['frequency'] > 5:
        return "Champions"
    elif row['frequency'] > 3:
        return "Loyal Customers"
    elif row['recency'] > 60:
        return "Dormant Customers"
    elif row['tickets'] > 3 and row['monetary'] > 1000:
        return "High Value Unhappy"
    elif row['sessions_30d'] < 2:
        return "Low Engagement"
    else:
        return "At Risk"

rfm['segment_name'] = rfm.apply(segment, axis=1)
``
